In [1]:
import csv
import numpy as np
import pandas as pd

In [2]:
problem = []

def parse_dataset():
    dates = []
    codes = []
    names = []
    volumes = []
    closes = []

    with open('train.csv', 'r') as file:
        csv_reader = csv.reader(file)
        next(csv_reader)

        for row in csv_reader:
            date = row[0]
            code = row[1]
            name = row[2]
            volume = int(row[3])
            close = int(row[7])

            if volume == 0:
                problem.append(code)
            dates.append(date)
            codes.append(code)
            names.append(name)
            volumes.append(volume)
            closes.append(close)

    return dates, codes, names, volumes, closes

In [3]:
def calculate_rsi(prices, period):
    # 가격 데이터를 기반으로 RSI 계산
    changes = []
    for i in range(1, 16):
        change = prices[i] - prices[i-1]
        changes.append(change)
    
    gains = [change for change in changes if change >= 0]
    losses = [-change for change in changes if change < 0]
    
    avg_gain = sum(gains) / period
    avg_loss = sum(losses) / period
    
    for i in range(period, 16):
        change = changes[i-1]
        if change >= 0:
            avg_gain = (avg_gain * (period - 1) + change) / period
            avg_loss = (avg_loss * (period - 1)) / period
        else:
            avg_gain = (avg_gain * (period - 1)) / period
            avg_loss = (avg_loss * (period - 1) - change) / period
    
    if avg_loss != 0:
        rsi = 100 - (100 / (1 + (avg_gain / avg_loss)))
    else:
        rsi = 100
    
    return rsi

In [4]:
def calculate_sharpe_ratio(returns, risk_free_rate=0.035):
    alpha = 0.2
    # 샤프 지수를 계산합니다.
    excess_returns = returns - risk_free_rate
    data = np.array(excess_returns)
    weighted_average = np.zeros_like(data)
    weighted_average[0] = data[0]
    for t in range(1, 16):
        weighted_average[t] = (1 - alpha) * data[t] + alpha * weighted_average[t - 1]
    mean_excess_return = np.mean(weighted_average)
    std_excess_return = np.std(weighted_average)
    sharpe_ratio = mean_excess_return / std_excess_return
    
    return sharpe_ratio

In [5]:
dates, codes, names, volumes, closes = parse_dataset()

sharpe_dict = {}
rsi_dict = {}
period = 15
sorted_codes = []
[sorted_codes.append(code) for code in codes if code not in sorted_codes]

for code in sorted_codes:
    
    index_list = [i for i, x in enumerate(codes) if x == code][:-15]
    code_closes = [closes[i] for i in index_list]
    # 종가를 기반으로 수익률 계산
    returns = np.diff(code_closes) / code_closes[:-1]
    returns = np.concatenate([[0], returns])
    
    sharpe = calculate_sharpe_ratio(returns, risk_free_rate=0.035)
    sharpe_dict[code] = sharpe

    rsi = calculate_rsi(code_closes, period)
    rsi_dict[code] = rsi

In [6]:
sorted_rsi = sorted(rsi_dict.items(), key=lambda x: x[1], reverse=True)
sorted_sharpe = sorted(sharpe_dict.items(), key=lambda x: x[1], reverse=True)

# sorted_sharpe와 sorted_rsi를 활용하여 데이터프레임 생성
df_sharpe = pd.DataFrame(sorted_sharpe, columns=['종목코드', '순위_sharpe'])
df_rsi = pd.DataFrame(sorted_rsi, columns=['종목코드', '순위_rsi'])

# 두 데이터프레임을 '종목코드'를 기준으로 합치기
merged_data = pd.merge(df_sharpe, df_rsi, on='종목코드')

In [7]:
#랭킹 합치기
combined_ranking = {}
for rank, (code, _) in enumerate(sorted_rsi):
    combined_ranking[code] = rank + combined_ranking.get(code, 0)

for rank, (code, _) in enumerate(sorted_sharpe):
    combined_ranking[code] = rank + combined_ranking.get(code, 0)

# 합친 랭킹으로 분류
sorted_combined = sorted(combined_ranking.items(), key=lambda x: x[1])

with open('baseline_submission.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['종목코드', '순위'])

    for rank, (code, _) in enumerate(sorted_combined, start=1):
        writer.writerow([code, rank])

In [8]:
dates, codes, names, volumes, opens, highs, lows, closes = parse_dataset()

sharpe_dict = {}

count=1
for code in set(codes):
    index_list = [i for i, x in enumerate(codes) if x == code][:-15]
    code_closes = [closes[i] for i in index_list]

    # RSI 계산
    rsi = calculate_rsi(code_closes,period)
    if rsi <= 15:
        rsi += 20
    if rsi >= 85:
        rsi -= 20

    # 기간 동안의 투자 수익률 계산
    returns = calculate_returns(code_closes)

    # 샤프 지수 계산
    sharpe_ratio = calculate_sharpe_ratio(returns)

    # ARIMA 모델을 사용하여 15일 동안의 미래 수익 예측
    forecast_returns = predict_returns(code_closes, forecast_steps=15)

    # 15일 동안의 미래 예측 수익을 기반으로 종목의 스코어 계산
    forecast_mean = np.mean(forecast_returns)
    score = -rsi + sharpe_ratio + forecast_mean
    count+=1
    sharpe_dict[code] = score

sorted_sharpe = sorted(sharpe_dict.items(), key=lambda x: x[1], reverse=True)

ValueError: not enough values to unpack (expected 8, got 5)

In [ ]:
# 정렬된 종합점수를 기준으로 종목 랭킹 작성
with open('baseline_submission.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['종목코드', '순위'])

    for rank, (code, _) in enumerate(sorted_combined, start=1):
        writer.writerow([code, rank])

In [ ]:
#거래정지 종목들 중 상위200위 하위200위에 들어가는 종목들 분류
data = {}
with open('baseline_submission.csv', newline='') as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:
        code = row[0]
        rank = int(row[1])
        data[code] = rank
    
i = 1
s = 1

for code in sorted(problem):
    if code in data:
        rank = data[code]
        test1 = 200 + i
        test2 = 1801 - s
        before = next(key for key, value in data.items() if value == rank)
        if rank <= 200:
            after = next(key for key, value in data.items() if value == test1)
            data[before], data[after] = data[after], data[before]
            i += 1
            
        elif rank >= 1801:
            after = next(key for key, value in data.items() if value == test2)
            data[before], data[after] = data[after], data[before]
            s += 1

            
sorted_data = dict(sorted(data.items(), key=lambda x: x[1]))

for i, (code, rank) in enumerate(sorted_data.items(), start=1):
    sorted_data[code] = i

with open("adjusted_submission.csv", 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['종목코드', '순위'])

    for code, rank in sorted_data.items():
        writer.writerow([code, rank])